# Import libraries and load data

### Purpose

This notebook uses unsupervised machine learning to identify U.S. states
with similar economic and labor-market structures.

The notebook loads the cleaned BEA and BLS datasets prepared in the
data-collection notebook. It then validates and combines the datasets
before performing scaling, dimensionality reduction, and clustering.

The analysis does not use a predefined state-category target.
Cluster membership will be discovered from the economic features.

## Step 1A — Import libraries

In [ ]:
# ============================================================
# STANDARD LIBRARIES
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd


# ============================================================
# VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# PREPROCESSING AND DIMENSIONALITY REDUCTION
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ============================================================
# CLUSTERING MODELS
# ============================================================

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering
)

from sklearn.mixture import GaussianMixture


# ============================================================
# CLUSTER EVALUATION
# ============================================================

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)


# ============================================================
# HIERARCHICAL CLUSTERING
# ============================================================

from scipy.cluster.hierarchy import (
    linkage,
    dendrogram
)


# ============================================================
# NOTEBOOK SETTINGS
# ============================================================

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

RANDOM_STATE = 42

print("Libraries imported successfully.")

### Interpretation

The imported libraries support five major tasks:

1. pandas and NumPy manage and transform the datasets.
2. Matplotlib and Seaborn create analytical visualizations.
3. StandardScaler places features on comparable scales.
4. PCA reduces the number of economic dimensions.
5. K-Means, hierarchical clustering, and Gaussian Mixture Models
   identify groups of economically similar states.

No model has been trained at this stage.

## Step 1B — Define project paths

In [ ]:
# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Processed-data folder:", PROCESSED_DIR.resolve())
print("Figure folder:", FIGURE_DIR.resolve())
print("Table folder:", TABLE_DIR.resolve())

## Step 1C — Check available processed files

In [ ]:
# ============================================================
# INSPECT AVAILABLE PROCESSED FILES
# ============================================================

available_files = sorted(
    PROCESSED_DIR.glob("*.csv")
)

if not available_files:
    raise FileNotFoundError(
        f"No CSV files were found in:\n"
        f"{PROCESSED_DIR.resolve()}"
    )

file_inventory = pd.DataFrame({
    "File": [
        file.name for file in available_files
    ],
    "Size_KB": [
        round(file.stat().st_size / 1024, 2)
        for file in available_files
    ]
})

display(file_inventory)

## Step 1D — Load the cleaned source datasets

In [ ]:
from pathlib import Path
import pandas as pd

BEA_DATA_DIR = Path("data") / "raw" / "bea"

gdp = pd.read_csv(
    BEA_DATA_DIR / "bea_SAGDP9_raw.csv",
    dtype={"GeoFIPS": "string"},
    low_memory=False
)

income = pd.read_csv(
    BEA_DATA_DIR / "bea_SAINC1_raw.csv",
    dtype={"GeoFIPS": "string"},
    low_memory=False
)

economic_profile = pd.read_csv(
    BEA_DATA_DIR / "bea_SAINC30_raw.csv",
    dtype={"GeoFIPS": "string"},
    low_memory=False
)

print("gdp:", sagdp9.shape)
print("income:", sainc1.shape)
print("economic_profile:", sainc30.shape)

In [ ]:
DATA_DIR = Path("data") / "raw" 
industry= pd.read_csv(DATA_DIR / "bls_qcew_industry_employment_2015_2025_raw.csv")
print(industry.shape)
industry.head(2)

## Load all four datasets

In [ ]:
datasets = {
    "SAGDP9": gdp,
    "SAINC1": income,
    "SAINC30": economic_profile,
    "BLS_Industry": industry
}

for dataset_name, dataframe in datasets.items():

    print(f"\n{dataset_name}")
    print("-" * 60)

    display(dataframe.head(3))

In [ ]:
# ============================================================
# LOADING SUMMARY
# ============================================================

loading_summary = pd.DataFrame([
    {
        "Dataset": dataset_name,
        "Rows": dataframe.shape[0],
        "Columns": dataframe.shape[1],
        "Missing_Cells": int(
            dataframe.isna().sum().sum()
        ),
        "Duplicate_Rows": int(
            dataframe.duplicated().sum()
        )
    }
    for dataset_name, dataframe in datasets.items()
])

display(loading_summary)